# Analysing Open Source Contributor Activity

**Author:** Owolabi Samuel Ogunlalu  
**Goal:** Understand contributor behaviour patterns in an open source project — who contributes, when, and how activity evolves over time.

---

Understanding contributor activity is essential for community managers. This notebook explores patterns that inform decisions around onboarding, mentorship, and contributor pathway design.

In practice, data like this comes from the **GitHub API** or repository analytics exports.

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

COLORS = ['#2E75B6', '#28A745', '#FD7E14', '#DC3545', '#6F42C1']
print('Ready.')

## 2. Simulate Contributor Dataset

This simulates a year of activity logs for a mid-sized open source project with ~80 contributors.

In [ ]:
np.random.seed(7)

n_contributors = 80
contributor_ids = [f'user_{i:03d}' for i in range(1, n_contributors + 1)]

# Contribution types
contrib_types = ['commit', 'pull_request', 'issue_comment', 'issue_opened', 'review']

# Generate random activity — more active contributors make more commits
activity_weights = np.random.pareto(1.5, n_contributors) + 1  # Pareto: realistic open source distribution
activity_weights = (activity_weights / activity_weights.sum() * 2000).astype(int).clip(min=1)

rows = []
for uid, weight in zip(contributor_ids, activity_weights):
    for _ in range(weight):
        rows.append({
            'contributor': uid,
            'type': np.random.choice(contrib_types, p=[0.40, 0.20, 0.25, 0.10, 0.05]),
            'month': np.random.choice(range(1, 13)),
            'week':  np.random.choice(range(1, 53)),
            'day_of_week': np.random.choice(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'],
                                             p=[0.18,0.19,0.18,0.18,0.17,0.05,0.05])
        })

df = pd.DataFrame(rows)
print(f'Total activity events: {len(df):,}')
print(f'Unique contributors  : {df["contributor"].nunique()}')
df.head()

## 3. Contribution Type Breakdown

In [ ]:
type_counts = df['type'].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(type_counts.index, type_counts.values, color=COLORS, alpha=0.85)

for bar, val in zip(bars, type_counts.values):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)

ax.set_title('Contribution Activity by Type', fontweight='bold', fontsize=12)
ax.set_xlabel('Number of Events')
ax.set_facecolor('#FDFDFD')
fig.patch.set_facecolor('#F8F9FA')
plt.tight_layout()
plt.savefig('contribution_types.png', bbox_inches='tight')
plt.show()

## 4. Monthly Activity Trend

In [ ]:
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('month').size().reindex(range(1, 13), fill_value=0)

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(month_labels, monthly.values, alpha=0.25, color=COLORS[0])
ax.plot(month_labels, monthly.values, color=COLORS[0], linewidth=2.5, marker='o', markersize=6)

for i, v in enumerate(monthly.values):
    ax.annotate(str(v), (month_labels[i], v), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8)

ax.set_title('Total Contributor Activity by Month', fontweight='bold', fontsize=12)
ax.set_ylabel('Activity Events')
ax.set_facecolor('#FDFDFD')
fig.patch.set_facecolor('#F8F9FA')
plt.tight_layout()
plt.savefig('monthly_activity.png', bbox_inches='tight')
plt.show()

## 5. Top 15 Contributors (Pareto Effect)

In [ ]:
top_15 = df['contributor'].value_counts().head(15)
total  = len(df)
top_15_pct = (top_15.sum() / total * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(top_15)), top_15.values, color=COLORS[0], alpha=0.85)
ax.set_xticks(range(len(top_15)))
ax.set_xticklabels(top_15.index, rotation=35, ha='right', fontsize=9)
ax.set_title(f'Top 15 Contributors  |  Responsible for {top_15_pct}% of all activity',
             fontweight='bold', fontsize=11)
ax.set_ylabel('Total Contributions')
ax.set_facecolor('#FDFDFD')
fig.patch.set_facecolor('#F8F9FA')
plt.tight_layout()
plt.savefig('top_contributors.png', bbox_inches='tight')
plt.show()

print(f'Top 15 contributors account for {top_15_pct}% of total activity.')
print('This is typical of the Pareto distribution seen in open source communities.')

## 6. Activity by Day of Week

In [ ]:
day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
day_counts = df['day_of_week'].value_counts().reindex(day_order)

fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = [COLORS[0] if d not in ['Sat', 'Sun'] else COLORS[2] for d in day_order]
ax.bar(day_order, day_counts.values, color=bar_colors, alpha=0.85)

ax.set_title('Contribution Activity by Day of Week', fontweight='bold', fontsize=12)
ax.set_ylabel('Activity Events')
ax.set_facecolor('#FDFDFD')
fig.patch.set_facecolor('#F8F9FA')

from matplotlib.patches import Patch
legend = [Patch(color=COLORS[0], label='Weekday'), Patch(color=COLORS[2], label='Weekend')]
ax.legend(handles=legend, fontsize=9)

plt.tight_layout()
plt.savefig('activity_by_day.png', bbox_inches='tight')
plt.show()

## 7. Community Manager Insights

These patterns directly inform community strategy:

**Pareto distribution in contributions**  
A small group of core contributors drives the majority of activity. Community managers should identify these members early, reduce friction for them, and create clear pathways to maintainer roles.

**Weekday concentration**  
Most activity happens Mon to Thu. Scheduling collaboration cafes, office hours, and review sessions on these days maximises attendance and responsiveness.

**Monthly dips**  
Activity dips in certain months (often summer and December). Proactive community managers plan lower-effort engagement — async threads, documentation sprints — around these periods to maintain momentum.

**Contribution type distribution**  
If commits vastly outnumber reviews and comments, the community may lack a review culture. Running structured PR review sessions and recognising reviewers publicly helps balance the workload.

---

## 8. Next Steps

- Connect to the GitHub API using `PyGithub` to pull real contributor data
- Add cohort retention analysis: what % of first-time contributors return?
- Build a contributor health score combining frequency, recency, and diversity of contribution types

---

*Notebook by Owolabi Samuel Ogunlalu.*